# Exploração inicial do PublicHearingBR

Este notebook apresenta o corpus principal do PublicHearingBR para sumarização de documentos longos. O estudo de referência descreve 206 audiências públicas brasileiras, cada uma com uma transcrição, uma matéria jornalística e metadados estruturados.

A fonte é o [paper PublicHearingBR](../2410.07495v2.pdf) e o [dataset no Hugging Face](https://huggingface.co/datasets/unicamp-dl/PublicHearingBR).

> Escopo: usamos somente "PublicHearingBR_LDS.jsonl". O arquivo "PublicHearingBR_NLI.jsonl" é um conjunto derivado para inferência em linguagem natural e avaliação de alucinação. Por isso, não responde à pergunta desta exploração sobre a composição do corpus de sumarização.

## 1. Download e carregamento

O download seleciona explicitamente o JSONL do corpus LDS. Isso evita carregar o NLI por engano e contorna a incompatibilidade de esquemas entre os dois arquivos no viewer público do Hugging Face.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

working_dir = Path.cwd().resolve()
project_root = working_dir if (working_dir / 'src').exists() else working_dir.parent
sys.path.insert(0, str(project_root / 'src'))

from ideias_em_rede.data.download import download_lds_dataset
from ideias_em_rede.data.load import load_jsonl

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
dataset_path = download_lds_dataset()
dataset_frame = load_jsonl(dataset_path)

expected_columns = {'id', 'transcricao', 'materia', 'metadados'}
assert len(dataset_frame) == 206, 'O LDS deve conter 206 registros.'
assert set(dataset_frame.columns) == expected_columns, 'O esquema do LDS mudou.'
assert dataset_frame['id'].is_unique, 'Os IDs devem ser únicos.'

display(Markdown(f"Arquivo carregado: `{dataset_path.name}`<br>Registros: **{len(dataset_frame)}**<br>IDs únicos: **{dataset_frame['id'].nunique()}**"))

## 2. Estrutura de cada audiência

Cada registro conecta o documento longo à sua referência jornalística e à informação estruturada extraída da matéria.

### Esqueleto de um registro JSONL

O arquivo é JSON Lines: cada linha representa uma audiência com esta estrutura. Os valores abaixo são ilustrativos e foram reduzidos para deixar visível apenas o formato.

```json
{
  "id": 1,
  "transcricao": "texto longo com os turnos da audiência...",
  "materia": "texto da matéria jornalística...",
  "metadados": {
    "assunto": "tema principal da audiência",
    "envolvidos": [
      {
        "nome": "Nome da pessoa",
        "cargo": "Cargo ou vínculo institucional",
        "opinioes": [
          "Posicionamento citado na matéria jornalística."
        ]
      }
    ]
  }
}
```

"metadados" organiza a cobertura da matéria. Para conferir a fala original, deve-se consultar "transcricao".

In [ ]:
field_frame = pd.DataFrame(
    [
        ('id', 'Identificador sequencial da audiência.'),
        ('transcricao', 'Transcrição completa da audiência pública; é o documento longo.'),
        ('materia', 'Matéria jornalística sobre a audiência; é a referência resumida.'),
        ('metadados', 'Assunto e participantes, com cargos e opiniões citadas na matéria.'),
    ],
    columns=['Campo', 'Significado'],
)
display(field_frame)

### Prévia dos temas das audiências

O campo "metadados["assunto"]" registra o tema principal associado a cada audiência. A tabela mostra uma amostra fixa dos dez primeiros registros.

In [ ]:
topic_preview_frame = pd.DataFrame(
    {
        'ID': dataset_frame['id'],
        'Assunto': dataset_frame['metadados'].map(lambda metadata: metadata['assunto']),
    }
)
display(topic_preview_frame.head(10))

In [ ]:
sample_record = dataset_frame.iloc[0]
sample_metadata = sample_record['metadados']
sample_participants = pd.DataFrame(sample_metadata['envolvidos'])
sample_overview = pd.DataFrame(
    {
        'ID': [sample_record['id']],
        'Assunto': [sample_metadata['assunto']],
        'Palavras na transcrição': [len(sample_record['transcricao'].split())],
        'Palavras na matéria': [len(sample_record['materia'].split())],
        'Participantes na matéria': [len(sample_participants)],
    }
)
display(sample_overview)
display(sample_participants[['nome', 'cargo', 'opinioes']].head(3))

## 3. Dois exemplos práticos do LDS

Os registros 1 e 2 são usados para tornar concreta a estrutura do arquivo. Cada registro é um objeto com uma transcrição contínua, uma matéria jornalística e metadados. Os participantes dos metadados são as pessoas relevantes citadas na matéria; eles não formam, necessariamente, a lista completa de todas as pessoas que aparecem na transcrição.

In [ ]:
example_ids = [1, 2]

for record_id in example_ids:
    record = dataset_frame.loc[dataset_frame['id'].eq(record_id)].iloc[0]
    metadata = record['metadados']
    participant_frame = pd.DataFrame(metadata['envolvidos']).copy()
    participant_frame['opinion_count'] = participant_frame['opinioes'].map(len)
    participant_frame['opinion_preview'] = participant_frame['opinioes'].map(
        lambda opinions: opinions[0][:220] if opinions else ''
    )
    example_frame = pd.DataFrame(
        {
            'ID': [record['id']],
            'Assunto': [metadata['assunto']],
            'Palavras na transcrição': [len(record['transcricao'].split())],
            'Palavras na matéria': [len(record['materia'].split())],
            'Participantes nos metadados': [len(participant_frame)],
            'Opiniões nos metadados': [participant_frame['opinion_count'].sum()],
        }
    )
    display(Markdown(f'### Registro {record_id}'))
    display(example_frame)
    display(participant_frame[['nome', 'cargo', 'opinion_count', 'opinion_preview']])

### Como a transcrição é organizada

"transcricao" é uma única string, não uma tabela pronta de falas. As mudanças de turno aparecem no início de linhas com rótulos como "O SR." e "A SRA.". Portanto, a mesma pessoa pode falar, ser interrompida por outra e voltar a falar depois. A célula a seguir separa esses blocos, mostra a ordem dos turnos e aponta quem reaparece.

In [ ]:
import re

turn_marker_pattern = re.compile(r'(?m)^(?:O SR[.]|A SRA[.])')

def split_turns(transcript):
    turn_starts = list(turn_marker_pattern.finditer(transcript))
    return [
        transcript[match.start(): turn_starts[index + 1].start()].strip()
        if index + 1 < len(turn_starts)
        else transcript[match.start():].strip()
        for index, match in enumerate(turn_starts)
    ]

def extract_speaker_label(turn):
    first_line = turn.splitlines()[0]
    parenthetical_delimiter = re.search(r'[)] *- *', first_line)
    if parenthetical_delimiter:
        return first_line[: parenthetical_delimiter.start() + 1].strip()
    generic_delimiter = re.search(r'- +', first_line)
    if generic_delimiter:
        return first_line[: generic_delimiter.start()].strip()
    return first_line.strip()

def build_turn_frame(transcript):
    turn_rows = []
    for turn_number, turn in enumerate(split_turns(transcript), start=1):
        speaker = extract_speaker_label(turn)
        speech = turn[len(speaker):].lstrip(' -').lstrip()
        turn_rows.append(
            {
                'turn': turn_number,
                'speaker': speaker,
                'turn_word_count': len(speech.split()),
                'speech_preview': speech[:360],
            }
        )
    return pd.DataFrame(turn_rows)

for record_id in example_ids:
    record = dataset_frame.loc[dataset_frame['id'].eq(record_id)].iloc[0]
    turn_frame = build_turn_frame(record['transcricao'])
    speaker_summary_frame = (
        turn_frame.groupby('speaker', as_index=False)
        .agg(
            turn_count=('turn', 'size'),
            total_turn_words=('turn_word_count', 'sum'),
            mean_words_per_turn=('turn_word_count', 'mean'),
        )
        .sort_values(['turn_count', 'total_turn_words'], ascending=False)
    )
    total_turn_words = turn_frame['turn_word_count'].sum()
    mean_turn_words = turn_frame['turn_word_count'].mean()
    repeated_speaker_count = speaker_summary_frame['turn_count'].gt(1).sum()
    display(Markdown(f'### Sequência de turnos do registro {record_id}'))
    display(Markdown(f'Foram identificados **{len(turn_frame)} turnos**, **{repeated_speaker_count} rótulos de falante repetidos**, **{total_turn_words:,.0f} palavras nos turnos** e média de **{mean_turn_words:,.1f} palavras por turno**.'))
    display(turn_frame[['turn', 'speaker', 'turn_word_count', 'speech_preview']].head(12))
    display(speaker_summary_frame.round({'mean_words_per_turn': 1}))

### Exemplos de funções discursivas da audiência 1

As anotações abaixo usam frases reais das audiências 1 e 2. Elas não são rótulos oficiais do PublicHearingBR. Nesta proposta, "claim" é uma afirmação ou compromisso que pode ser discutido, "premise" é uma razão ou evidência usada para sustentar uma afirmação e "general_information" oferece contexto ou procedimento sem desempenhar, naquele trecho, um papel argumentativo central.

A unidade é uma frase completa, delimitada aqui pelo ponto final. Em uma etapa futura, a segmentação automática deve tratar abreviações como "Sr." e "V.Exa." com um segmentador de sentenças em português, e não com uma simples divisão por ponto.

In [ ]:
illustrative_sentence_record_one_frame = pd.DataFrame(
    [
        {
            'record_id': 1,
            'speaker': 'Lucas Redecker',
            'sentence': 'O autor do requerimento desta audiência ainda não chegou.',
            'label': 'general_information',
            'interpretation': 'Informação factual sobre o andamento da sessão.',
        },
        {
            'record_id': 1,
            'speaker': 'Lucas Redecker',
            'sentence': 'Passo, de imediato, a palavra ao Sr. Eli Vieira Araujo Júnior, que tem o tempo de até 15 minutos para a sua exposição.',
            'label': 'general_information',
            'interpretation': 'Instrução de procedimento que organiza a ordem das falas.',
        },
        {
            'record_id': 1,
            'speaker': 'Lucas Redecker',
            'sentence': 'Todos os que estão nos acompanhando presencialmente podem notar que o volume do nosso microfone é mais alto do que o daqueles que estão expondo de forma virtual.',
            'label': 'premise',
            'interpretation': 'Observação que justifica o pedido de silêncio feito logo em seguida.',
        },
        {
            'record_id': 1,
            'speaker': 'Eli Vieira Araujo Júnior',
            'sentence': 'Esses termos incluem em si expressões e atos já codificados como crimes em lei.',
            'label': 'premise',
            'interpretation': 'Razão apresentada para sustentar a posição do orador sobre a aplicação da lei.',
        },
        {
            'record_id': 1,
            'speaker': 'Eli Vieira Araujo Júnior',
            'sentence': 'Isso — é claro — viola diversos princípios do Estado Democrático de Direito.',
            'label': 'claim',
            'interpretation': 'Conclusão normativa e contestável do orador.',
        },
        {
            'record_id': 1,
            'speaker': 'Eli Vieira Araujo Júnior',
            'sentence': 'É a lei que deve mandar, e não a opinião política em voga no momento, entre setores influentes da imprensa, da academia ou das instituições.',
            'label': 'claim',
            'interpretation': 'Tese normativa que expressa o posicionamento central do orador.',
        },
    ]
)
record_one_transcript = dataset_frame.loc[dataset_frame['id'].eq(1), 'transcricao'].iloc[0]
assert illustrative_sentence_record_one_frame['sentence'].map(record_one_transcript.__contains__).all()
display(illustrative_sentence_record_one_frame)


### Exemplos de funções discursivas da audiência 2

A audiência 2 recebe a mesma leitura ilustrativa. Os exemplos mantêm a distinção entre afirmações, razões e informações de procedimento, mas a interpretação continua dependente do contexto do turno e da discussão completa.

In [ ]:
illustrative_sentence_record_two_frame = pd.DataFrame(
    [
        {
            'record_id': 2,
            'speaker': 'Rodrigo de Castro',
            'sentence': 'Na sequência, a palavra será facultada aos autores dos requerimentos por 5 minutos.',
            'label': 'general_information',
            'interpretation': 'Regra de condução da sessão; informa o procedimento sem defender uma tese.',
        },
        {
            'record_id': 2,
            'speaker': 'Luiz Carlos Ciocchi',
            'sentence': 'Com os dados que existem, nós tentamos simular o evento, e não conseguimos.',
            'label': 'premise',
            'interpretation': 'Evidência apresentada para sustentar a avaliação sobre a atuação do ONS.',
        },
        {
            'record_id': 2,
            'speaker': 'Luiz Carlos Ciocchi',
            'sentence': 'Eu diria o seguinte ao senhor a respeito daquele ponto no qual o senhor disse que o ONS falhou: até esse ponto e exatamente nesse ponto, eu garanto ao senhor que o ONS não falhou.',
            'label': 'claim',
            'interpretation': 'Posicionamento explícito e contestável sobre o desempenho do ONS.',
        },
        {
            'record_id': 2,
            'speaker': 'Luiz Carlos Ciocchi',
            'sentence': 'A causa dessa diferença nos dados será objeto do RAP, relatório que está sendo elaborado.',
            'label': 'general_information',
            'interpretation': 'Informação sobre o encaminhamento técnico do relatório, sem apoiar diretamente uma tese.',
        },
        {
            'record_id': 2,
            'speaker': 'Alexandre Silveira',
            'sentence': 'Se houver legalidade para que isso aconteça, nós vamos avançar nesse sentido.',
            'label': 'claim',
            'interpretation': 'Compromisso condicional com uma ação futura do governo.',
        },
        {
            'record_id': 2,
            'speaker': 'Luiz Carlos Ciocchi',
            'sentence': 'Quanto à hidrelétrica, São Francisco já está operando, por conta da época da seca, na sua vazão e geração mínima — portanto, não poderia ser reduzida.',
            'label': 'premise',
            'interpretation': 'Justificativa técnica para explicar por que a fonte não poderia ser reduzida.',
        },
    ]
)
record_two_transcript = dataset_frame.loc[dataset_frame['id'].eq(2), 'transcricao'].iloc[0]
assert illustrative_sentence_record_two_frame['sentence'].map(record_two_transcript.__contains__).all()
display(illustrative_sentence_record_two_frame)

### Mapa de turnos e extensão das falas

Cada bolha representa uma fala. A posição horizontal é a ordem do turno, a vertical identifica o falante e a área da bolha representa o número de palavras daquele turno. Aqui, intensidade significa extensão textual da fala, e não volume, emoção ou importância política. Para não criar uma legenda excessivamente longa, os oito falantes com mais turnos recebem cores próprias e os demais aparecem como "Outros".

In [ ]:
from matplotlib.lines import Line2D

plot_record_id = 1
max_legend_speakers = 8

record = dataset_frame.loc[dataset_frame['id'].eq(plot_record_id)].iloc[0]
turn_frame = build_turn_frame(record['transcricao'])
speaker_counts = turn_frame['speaker'].value_counts()
top_speakers = speaker_counts.head(max_legend_speakers).index

def shorten_speaker_name(speaker, limit=42):
    speaker_name = speaker.replace('O SR. ', '').replace('A SRA. ', '')
    return speaker_name if len(speaker_name) <= limit else f'{speaker_name[:limit - 3]}...'

plot_frame = turn_frame.copy()
plot_frame['speaker_text'] = plot_frame['speaker'].map(shorten_speaker_name)
speaker_order = [shorten_speaker_name(speaker) for speaker in speaker_counts.index]
top_group_labels = [shorten_speaker_name(speaker) for speaker in top_speakers]
plot_frame['speaker_group'] = plot_frame['speaker_text'].where(
    plot_frame['speaker'].isin(top_speakers),
    'Outros',
)
plot_frame['speaker_label'] = pd.Categorical(
    plot_frame['speaker_text'],
    categories=speaker_order,
    ordered=True,
)
palette = dict(zip(top_group_labels, sns.color_palette('tab10', n_colors=len(top_group_labels))))
palette['Outros'] = '#bdbdbd'

minimum_bubble_area = 35
maximum_bubble_area = 700
minimum_turn_words = plot_frame['turn_word_count'].min()
maximum_turn_words = plot_frame['turn_word_count'].max()

def scale_bubble_area(word_count):
    if maximum_turn_words == minimum_turn_words:
        return (minimum_bubble_area + maximum_bubble_area) / 2
    return minimum_bubble_area + (word_count - minimum_turn_words) * (
        maximum_bubble_area - minimum_bubble_area
    ) / (maximum_turn_words - minimum_turn_words)

figure, (axis, legend_axis) = plt.subplots(
    ncols=2,
    figsize=(19, max(6, len(speaker_order) * 0.38)),
    gridspec_kw={'width_ratios': [2.8, 1.8]},
    constrained_layout=True,
)
legend_axis.set_axis_off()
sns.scatterplot(
    data=plot_frame,
    x='turn',
    y='speaker_label',
    hue='speaker_group',
    size='turn_word_count',
    sizes=(minimum_bubble_area, maximum_bubble_area),
    palette=palette,
    alpha=0.78,
    legend=False,
    ax=axis,
)
speaker_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        color='white',
        markerfacecolor=palette[label],
        label=label,
        markersize=9,
    )
    for label in [*top_group_labels, 'Outros']
]
speaker_legend = legend_axis.legend(
    handles=speaker_handles,
    title='Falantes mais recorrentes',
    bbox_to_anchor=(0, 1),
    loc='upper left',
    borderpad=1.1,
    labelspacing=1.15,
    handletextpad=0.9,
)
legend_axis.add_artist(speaker_legend)

word_count_values = sorted(
    {
        int(round(plot_frame['turn_word_count'].quantile(quantile)))
        for quantile in [0.25, 0.5, 0.75]
    }
)
word_count_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        color='white',
        markerfacecolor='#7f7f7f',
        alpha=0.78,
        label=f'{word_count} palavras',
        markersize=scale_bubble_area(word_count) ** 0.5,
    )
    for word_count in word_count_values
]
legend_axis.legend(
    handles=word_count_handles,
    title='Extensão da fala',
    bbox_to_anchor=(0, 0.32),
    loc='upper left',
    borderpad=1.1,
    labelspacing=1.25,
    handletextpad=1.1,
)
axis.set_title(f'Mapa de turnos da audiência {plot_record_id}')
axis.set_xlabel('Ordem do turno na transcrição')
axis.set_ylabel('Falante')
plt.show()

### Mapa de turnos da audiência 2

Aplicamos a mesma leitura à segunda audiência para comparar visualmente o papel dos participantes e a distribuição das falas.

In [ ]:
plot_record_id = 2
max_legend_speakers = 8

record = dataset_frame.loc[dataset_frame['id'].eq(plot_record_id)].iloc[0]
turn_frame = build_turn_frame(record['transcricao'])
speaker_counts = turn_frame['speaker'].value_counts()
top_speakers = speaker_counts.head(max_legend_speakers).index

plot_frame = turn_frame.copy()
plot_frame['speaker_text'] = plot_frame['speaker'].map(shorten_speaker_name)
speaker_order = [shorten_speaker_name(speaker) for speaker in speaker_counts.index]
top_group_labels = [shorten_speaker_name(speaker) for speaker in top_speakers]
plot_frame['speaker_group'] = plot_frame['speaker_text'].where(
    plot_frame['speaker'].isin(top_speakers),
    'Outros',
)
plot_frame['speaker_label'] = pd.Categorical(
    plot_frame['speaker_text'],
    categories=speaker_order,
    ordered=True,
)
palette = dict(zip(top_group_labels, sns.color_palette('tab10', n_colors=len(top_group_labels))))
palette['Outros'] = '#bdbdbd'
minimum_bubble_area = 35
maximum_bubble_area = 700
minimum_turn_words = plot_frame['turn_word_count'].min()
maximum_turn_words = plot_frame['turn_word_count'].max()

def scale_bubble_area_record_two(word_count):
    if maximum_turn_words == minimum_turn_words:
        return (minimum_bubble_area + maximum_bubble_area) / 2
    return minimum_bubble_area + (word_count - minimum_turn_words) * (
        maximum_bubble_area - minimum_bubble_area
    ) / (maximum_turn_words - minimum_turn_words)

figure, (axis, legend_axis) = plt.subplots(
    ncols=2,
    figsize=(19, max(6, len(speaker_order) * 0.38)),
    gridspec_kw={'width_ratios': [2.8, 1.8]},
    constrained_layout=True,
)
legend_axis.set_axis_off()
sns.scatterplot(
    data=plot_frame,
    x='turn',
    y='speaker_label',
    hue='speaker_group',
    size='turn_word_count',
    sizes=(minimum_bubble_area, maximum_bubble_area),
    palette=palette,
    alpha=0.78,
    legend=False,
    ax=axis,
)
speaker_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        color='white',
        markerfacecolor=palette[label],
        label=label,
        markersize=9,
    )
    for label in [*top_group_labels, 'Outros']
]
speaker_legend = legend_axis.legend(
    handles=speaker_handles,
    title='Falantes mais recorrentes',
    bbox_to_anchor=(0, 1),
    loc='upper left',
    borderpad=1.1,
    labelspacing=1.15,
    handletextpad=0.9,
)
legend_axis.add_artist(speaker_legend)
word_count_values = sorted(
    {
        int(round(plot_frame['turn_word_count'].quantile(quantile)))
        for quantile in [0.25, 0.5, 0.75]
    }
)
word_count_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        color='white',
        markerfacecolor='#7f7f7f',
        alpha=0.78,
        label=f'{word_count} palavras',
        markersize=scale_bubble_area_record_two(word_count) ** 0.5,
    )
    for word_count in word_count_values
]
legend_axis.legend(
    handles=word_count_handles,
    title='Extensão da fala',
    bbox_to_anchor=(0, 0.32),
    loc='upper left',
    borderpad=1.1,
    labelspacing=1.25,
    handletextpad=1.1,
)
axis.set_title(f'Mapa de turnos da audiência {plot_record_id}')
axis.set_xlabel('Ordem do turno na transcrição')
axis.set_ylabel('Falante')
plt.show()

### Como localizar a fala de uma pessoa

Primeiro, escolha o registro pelo "id". Depois, use um fragmento do nome como ele aparece no cabeçalho da transcrição. O nome em "metadados['envolvidos']" ajuda a escolher a pessoa, mas pode ter uma pequena variação em relação ao rótulo da fala: por exemplo, títulos e nomes do meio podem aparecer no texto transcrito.

In [ ]:
record_id = 2
speaker_fragment = 'ALEXANDRE SILVEIRA'

record = dataset_frame.loc[dataset_frame['id'].eq(record_id)].iloc[0]
turn_frame = build_turn_frame(record['transcricao'])
speaker_turns = turn_frame[
    turn_frame['speaker'].str.upper().str.contains(speaker_fragment, regex=False)
].copy()

display(Markdown(f'Falas encontradas para **{speaker_fragment}** no registro **{record_id}**: **{len(speaker_turns)}**.'))
display(speaker_turns[['turn', 'speaker', 'speech_preview']])

metadata_participants = pd.DataFrame(record['metadados']['envolvidos'])
display(metadata_participants[['nome', 'cargo', 'opinioes']])

## 4. Métricas do corpus

O paper enfatiza quatro dimensões: extensão das transcrições, extensão das matérias, taxa de compressão e riqueza dos metadados. Para manter a comparação intuitiva, contamos palavras por separação de espaços. Pequenas diferenças em relação à tabela do paper podem ocorrer se a tokenização original tiver sido diferente.

In [ ]:
def count_words(text):
    return len(str(text).split())

def count_opinions(metadata):
    return sum(len(person['opinioes']) for person in metadata['envolvidos'])

analysis_frame = dataset_frame.copy()
analysis_frame['transcript_word_count'] = analysis_frame['transcricao'].map(count_words)
analysis_frame['article_word_count'] = analysis_frame['materia'].map(count_words)
analysis_frame['compression_ratio_percent'] = (
    analysis_frame['article_word_count'] / analysis_frame['transcript_word_count'] * 100
)
analysis_frame['participant_count'] = analysis_frame['metadados'].map(
    lambda metadata: len(metadata['envolvidos'])
)
analysis_frame['opinion_count'] = analysis_frame['metadados'].map(count_opinions)

In [ ]:
paper_reference_frame = pd.DataFrame(
    [
        ('Transcrição', 'palavras', 18102),
        ('Matéria', 'palavras', 627),
        ('Matéria/transcrição', '%', 4.20),
        ('Participantes', 'quantidade', 5.2),
        ('Opiniões', 'quantidade', 10.7),
    ],
    columns=['Métrica', 'Unidade', 'Média reportada no paper'],
)
display(paper_reference_frame)

In [ ]:
metric_columns = [
    'transcript_word_count',
    'article_word_count',
    'compression_ratio_percent',
    'participant_count',
    'opinion_count',
]
summary_frame = analysis_frame[metric_columns].describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
).T
summary_frame = summary_frame.rename(
    columns={
        'count': 'Registros',
        'mean': 'Média',
        'std': 'Desvio padrão',
        'min': 'Mínimo',
        '5%': 'Q5%',
        '25%': 'Q25%',
        '50%': 'Mediana',
        '75%': 'Q75%',
        '95%': 'Q95%',
        'max': 'Máximo',
    }
)
summary_frame.index = [
    'Palavras na transcrição',
    'Palavras na matéria',
    'Matéria/transcrição (%)',
    'Participantes',
    'Opiniões',
]
display(summary_frame.round(2))

## 5. Visualizações

Os gráficos respondem a três perguntas diretas: quão longas são as fontes, quanto as matérias comprimem essas fontes e como participantes e opiniões se relacionam.

In [ ]:
figure, axis = plt.subplots(figsize=(10, 5))
sns.histplot(data=analysis_frame, x='transcript_word_count', bins=24, color='#2a6f97', ax=axis)
axis.axvline(analysis_frame['transcript_word_count'].median(), color='#d1495b', linestyle='--', label='Mediana')
axis.set_title('Distribuição do tamanho das transcrições')
axis.set_xlabel('Palavras na transcrição')
axis.set_ylabel('Número de audiências')
axis.legend()
plt.show()

In [ ]:
figure, axis = plt.subplots(figsize=(10, 5))
sns.histplot(data=analysis_frame, x='compression_ratio_percent', bins=20, color='#52b788', ax=axis)
axis.axvline(analysis_frame['compression_ratio_percent'].median(), color='#d1495b', linestyle='--', label='Mediana')
axis.set_title('Taxa de compressão das matérias jornalísticas')
axis.set_xlabel('Matéria como percentual da transcrição')
axis.set_ylabel('Número de audiências')
axis.legend()
plt.show()

### Média de opiniões por participantes

Cada barra representa as audiências com a mesma quantidade de participantes citados. O número no centro da barra é a média de opiniões estruturadas, e o rótulo n no eixo X informa quantas audiências compõem essa média. As hastes indicam um desvio-padrão acima e abaixo da média, ou seja, a variação entre as audiências do grupo. Barras mais altas indicam maior densidade média de opiniões registradas nas matérias; elas não representam necessariamente todos os participantes ou todas as falas da audiência.

In [ ]:
relationship_summary_frame = (
    analysis_frame
    .groupby('participant_count')
    .agg(
        mean_opinion_count=('opinion_count', 'mean'),
        std_opinion_count=('opinion_count', 'std'),
        hearing_count=('id', 'size'),
    )
    .reset_index()
    .sort_values('participant_count')
)

participant_labels = [
    f'{participant_count}\nn={hearing_count}'
    for participant_count, hearing_count in zip(
        relationship_summary_frame['participant_count'],
        relationship_summary_frame['hearing_count'],
    )
]

figure, axis = plt.subplots(figsize=(10, 6))
bars = axis.bar(
    participant_labels,
    relationship_summary_frame['mean_opinion_count'],
    color='#6a4c93',
    width=0.72,
)
bar_positions = [
    bar.get_x() + bar.get_width() / 2
    for bar in bars
]
axis.errorbar(
    bar_positions,
    relationship_summary_frame['mean_opinion_count'],
    yerr=relationship_summary_frame['std_opinion_count'],
    fmt='none',
    color='#38285a',
    capsize=4,
    linewidth=1.2,
)
axis.bar_label(
    bars,
    labels=[
        f'{mean_value:.1f}'
        for mean_value in relationship_summary_frame['mean_opinion_count']
    ],
    label_type='center',
    color='white',
    fontsize=10,
    fontweight='bold',
)
maximum_value = (
    relationship_summary_frame['mean_opinion_count']
    + relationship_summary_frame['std_opinion_count'].fillna(0)
).max()
axis.set_title('Média e variação de opiniões por participantes citados')
axis.set_xlabel('Participantes citados na matéria')
axis.set_ylabel('Opiniões estruturadas (média ± desvio-padrão)')
axis.set_ylim(0, maximum_value * 1.16)
axis.grid(axis='y', alpha=0.2)
plt.show()

## 6. Síntese

A síntese abaixo é calculada a partir do arquivo carregado e traduz os números em uma leitura direta do corpus.

In [ ]:
median_transcript = analysis_frame['transcript_word_count'].median()
median_article = analysis_frame['article_word_count'].median()
mean_compression = analysis_frame['compression_ratio_percent'].mean()
mean_participants = analysis_frame['participant_count'].mean()
mean_opinions = analysis_frame['opinion_count'].mean()

summary_text = f'''
- O corpus reúne **{len(analysis_frame)} audiências**, com mediana de **{median_transcript:,.0f} palavras** por transcrição.
- As matérias têm mediana de **{median_article:,.0f} palavras** e representam, em média, **{mean_compression:.2f}%** da extensão das transcrições.
- Cada matéria registra, em média, **{mean_participants:.1f} participantes** e **{mean_opinions:.1f} opiniões** estruturadas.
- Esses números confirmam que o desafio central do LDS é selecionar informação relevante em documentos longos, usando a matéria e os metadados como referências complementares.
'''
display(Markdown(summary_text))